# Random Forests & Ensemble Methods: Visual Guide

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/methods-algorithms/blob/master/notebooks/L04_rf.ipynb)

**Learning Objectives:**
- See why single decision trees are unstable (high variance)
- Understand how bootstrap sampling and aggregation reduce variance
- Visualize the difference between bagging (Random Forest) and boosting (AdaBoost, Gradient Boosting)
- Compare ensemble methods on accuracy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score

np.random.seed(42)

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

# ML color palette
ML_PURPLE = '#3333B2'
ML_BLUE = '#0066CC'
ML_ORANGE = '#FF7F0E'
ML_GREEN = '#2CA02C'
ML_RED = '#D62728'

print('Setup complete.')

## 1. The Problem with Single Trees

In [ ]:
# Shared dataset
X, y = make_classification(
    n_samples=200, n_features=2, n_redundant=0, n_informative=2,
    n_clusters_per_class=1, flip_y=0.1, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Decision boundary helper
def plot_decision_boundary(model, X, y, ax, title=''):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    cmap_bg = ListedColormap(['#cce0ff', '#ffe0cc'])
    ax.contourf(xx, yy, Z, alpha=0.4, cmap=cmap_bg)
    ax.scatter(X[:, 0], X[:, 1], c=[ML_BLUE if yi == 0 else ML_ORANGE for yi in y],
               edgecolors='white', s=40, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, alpha=0.3)

print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')

In [ ]:
# Visual 1: 5 trees on different bootstrap samples
colors = [ML_BLUE, ML_ORANGE, ML_GREEN, ML_RED, ML_PURPLE]
h = 0.05
x_min, x_max = X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5
y_min, y_max = X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))

fig, ax = plt.subplots(figsize=(10, 6))

for i in range(5):
    idx = np.random.RandomState(i).choice(len(X_train), size=len(X_train), replace=True)
    tree = DecisionTreeClassifier(max_depth=5, random_state=i)
    tree.fit(X_train[idx], y_train[idx])
    Z = tree.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)
    ax.contour(xx, yy, Z, levels=[0.5], colors=[colors[i]], linewidths=2, alpha=0.8)

ax.scatter(X_train[:, 0], X_train[:, 1],
           c=[ML_BLUE if yi == 0 else ML_ORANGE for yi in y_train],
           edgecolors='white', s=40, alpha=0.7)
ax.set_title('5 Trees on Different Bootstrap Samples \u2192 High Variance!')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.text(0.02, 0.02, 'Each tree produces a different boundary',
        transform=ax.transAxes, fontsize=11, color=ML_RED, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
ax.legend(handles=[mpatches.Patch(color=c, label=f'Tree {i+1}') for i, c in enumerate(colors)],
          loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Bootstrap Sampling

In [ ]:
# Visual 2: Bootstrap visualization
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

# Panel 0: Original dataset
axes[0].scatter(X_train[:, 0], X_train[:, 1],
                c=[ML_BLUE if yi == 0 else ML_ORANGE for yi in y_train],
                edgecolors='white', s=40, alpha=0.9)
axes[0].set_title(f'Original Data (n={len(X_train)})')
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')
axes[0].grid(True, alpha=0.3)

# Panels 1-3: Bootstrap samples
for k in range(1, 4):
    idx = np.random.RandomState(k).choice(len(X_train), size=len(X_train), replace=True)
    sampled_mask = np.zeros(len(X_train), dtype=bool)
    sampled_mask[np.unique(idx)] = True
    n_unique = sampled_mask.sum()

    # Faded points (not sampled)
    axes[k].scatter(X_train[~sampled_mask, 0], X_train[~sampled_mask, 1],
                    c=[ML_BLUE if yi == 0 else ML_ORANGE for yi in y_train[~sampled_mask]],
                    edgecolors='white', s=40, alpha=0.15)
    # Solid points (sampled)
    axes[k].scatter(X_train[sampled_mask, 0], X_train[sampled_mask, 1],
                    c=[ML_BLUE if yi == 0 else ML_ORANGE for yi in y_train[sampled_mask]],
                    edgecolors='white', s=40, alpha=0.9)
    axes[k].set_title(f'Bootstrap {k}\nUnique: {n_unique}/{len(X_train)} ({100*n_unique/len(X_train):.0f}%)')
    axes[k].set_xlabel('Feature 1')
    axes[k].set_ylabel('Feature 2')
    axes[k].grid(True, alpha=0.3)

plt.suptitle('Bootstrap Sampling: ~63% of Points Selected Each Time', fontsize=14, y=1.04)
plt.tight_layout()
plt.show()

## 3. Random Forest: Averaging Reduces Variance

In [ ]:
# Visual 3: Random Forest smooth decision boundary
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)

fig, ax = plt.subplots(figsize=(10, 6))
plot_decision_boundary(rf, X_train, y_train, ax,
                       title=f'Random Forest (100 Trees) \u2014 Test Acc: {test_acc:.3f}')
plt.tight_layout()
plt.show()

print(f'Train accuracy: {train_acc:.3f}')
print(f'Test accuracy:  {test_acc:.3f}')

In [ ]:
# Visual 4: Single tree vs RF side-by-side
single_tree = DecisionTreeClassifier(random_state=42)
single_tree.fit(X_train, y_train)
tree_test_acc = single_tree.score(X_test, y_test)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_decision_boundary(single_tree, X_train, y_train, axes[0],
                       title=f'Single Decision Tree\nTest Acc: {tree_test_acc:.3f}')
plot_decision_boundary(rf, X_train, y_train, axes[1],
                       title=f'Random Forest (100 Trees)\nTest Acc: {test_acc:.3f}')
plt.suptitle('Single Tree vs. Random Forest', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 4. How Many Trees?

In [ ]:
# Visual 5: OOB error vs n_estimators
n_trees_range = list(range(1, 201, 5))
oob_errors = []

for n in n_trees_range:
    rf_oob = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42)
    rf_oob.fit(X_train, y_train)
    oob_errors.append(1 - rf_oob.oob_score_)

converged_error = np.mean(oob_errors[-5:])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(n_trees_range, oob_errors, color=ML_PURPLE, linewidth=2)
ax.axhline(y=converged_error, color='gray', linestyle='--', linewidth=1.5, alpha=0.7)
ax.set_xlabel('Number of Trees')
ax.set_ylabel('OOB Error Rate')
ax.set_title('OOB Error vs. Number of Trees')

# Find where error stabilizes (first time within 0.005 of converged)
stable_idx = next((i for i, e in enumerate(oob_errors) if abs(e - converged_error) < 0.005), len(oob_errors) - 1)
stable_n = n_trees_range[stable_idx]
ax.annotate(f'Error stabilizes around {stable_n} trees',
            xy=(stable_n, oob_errors[stable_idx]),
            xytext=(stable_n + 30, oob_errors[stable_idx] + 0.02),
            fontsize=11, arrowprops=dict(arrowstyle='->', color='black'),
            bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
# Visual 6: Feature importance bar chart with 8 features
X_imp, y_imp = make_classification(n_samples=300, n_features=8, n_informative=4,
                                    n_redundant=2, random_state=42)
feature_names = [f'Feature {i+1}' for i in range(8)]
rf_imp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_imp.fit(X_imp, y_imp)

importances = rf_imp.feature_importances_
sorted_idx = np.argsort(importances)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh([feature_names[i] for i in sorted_idx], importances[sorted_idx],
               color=ML_BLUE, edgecolor='white')
for bar, val in zip(bars, importances[sorted_idx]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{val:.3f}', va='center', fontsize=11)
ax.set_xlabel('Importance')
ax.set_title('Feature Importance (Mean Decrease in Impurity)')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 6. Boosting: Sequential Correction

In [ ]:
# Visual 7: AdaBoost staged train/test error
ada = AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42, algorithm='SAMME')
ada.fit(X_train, y_train)

train_errors = [1 - score for score in ada.staged_score(X_train, y_train)]
test_errors = [1 - score for score in ada.staged_score(X_test, y_test)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(range(1, 201), train_errors, color=ML_BLUE, linewidth=2, label='Train Error')
ax.plot(range(1, 201), test_errors, color=ML_ORANGE, linewidth=2, label='Test Error')
ax.set_xlabel('Number of Estimators')
ax.set_ylabel('Error Rate')
ax.set_title('AdaBoost: Staged Train vs. Test Error')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Visual 8: Gradient Boosting with 3 learning rates
fig, ax = plt.subplots(figsize=(10, 6))

for lr, color in zip([0.01, 0.1, 0.5], [ML_BLUE, ML_ORANGE, ML_GREEN]):
    gb = GradientBoostingClassifier(n_estimators=200, learning_rate=lr,
                                     max_depth=3, random_state=42)
    gb.fit(X_train, y_train)
    test_errors = []
    for y_pred in gb.staged_predict(X_test):
        test_errors.append(1 - accuracy_score(y_test, y_pred))
    ax.plot(range(1, 201), test_errors, color=color, linewidth=2, label=f'lr={lr}')

ax.set_xlabel('Number of Estimators')
ax.set_ylabel('Test Error Rate')
ax.set_title('Gradient Boosting: Effect of Learning Rate')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Random Forest vs. Boosting

In [ ]:
# Visual 9: Model comparison bar chart
models = {
    'Single Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest (100)': RandomForestClassifier(n_estimators=100, random_state=42),
    'AdaBoost (50)': AdaBoostClassifier(n_estimators=50, random_state=42, algorithm='SAMME'),
    'Gradient Boosting (100)': GradientBoostingClassifier(n_estimators=100, random_state=42),
}
model_colors = [ML_RED, ML_BLUE, ML_ORANGE, ML_GREEN]

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    results[name] = model.score(X_test, y_test)

names = list(results.keys())
accs = list(results.values())

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(names, accs, color=model_colors, edgecolor='white', height=0.6)
for bar, acc in zip(bars, accs):
    ax.text(acc + 0.005, bar.get_y() + bar.get_height() / 2,
            f'{acc:.3f}', va='center', fontsize=12, fontweight='bold')
ax.set_xlabel('Test Accuracy')
ax.set_title('Model Comparison: Test Accuracy')
ax.set_xlim(min(accs) - 0.05, 1.0)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Model Comparison:')
for name, acc in results.items():
    print(f'  {name:30s} {acc:.3f}')

## Summary

**Key Takeaways:**
- **Single decision trees have high variance** -- different training samples produce very different boundaries
- **Random Forests reduce variance** by averaging many trees trained on bootstrap samples with random feature subsets
- **Boosting reduces bias** by training trees sequentially, each correcting the previous one's errors
- **Ensemble methods consistently outperform** single trees -- the choice between bagging and boosting depends on whether variance or bias dominates